# MPS State Preparation Resource Estimation Comparison

This notebook compares resource estimates for MPS (Matrix Product State) preparation
between the following implementations:

1. **QDK Chemistry Dense** (`MPSSequentialStatePreparation`): Uses CSD + Givens rotation
   decomposition.
2. **QDK Chemistry Sparse** (`MPSSparseStatePreparation`): Exploits U(1) block-sparsity
   of MPS tensors via permutation-based decomposition.
3. **Qualtran implementation from Rupprecht and Wölk**: Implements both dense and sparse state preparation methods.

Please download the code (Apache 2.0 license) and MPS data (CC BY 4.0 license) published by Felix Rupprecht
on [Zenodo](https://zenodo.org/records/20393500) `.

References
1. Felix Rupprecht and Sabine Wölk. (2026). Faster matrix product state preparation by
exploiting symmetry-induced block-sparsity.
https://arxiv.org/pdf/2605.28489. Zenodo: https://zenodo.org/records/20393500.

2. Dominic W. Berry et al. (2025). Rapid Initial-State Preparation for the Quantum Simulation of
Strongly Correlated Molecules. PRX Quantum 6, 020327.
https://doi.org/10.1103/PRXQuantum.6.020327.


In [1]:
import sys
import glob
from pathlib import Path
import numpy as np
from scipy.sparse import load_npz

from qdk_chemistry.data.mps_wavefunction import MPSWavefunction
from qdk_chemistry.algorithms.state_preparation import MPSSequentialStatePreparation
from qdk_chemistry.algorithms.state_preparation.mps_sparse import MPSSparseStatePreparation

## Load MPS tensors into MPSWavefunction
The `fe2s2-2_small` dataset contains 20 MPS tensors (compressed to bond dimension
≤ 1000).

In [2]:
COMPRESSED_PATH = Path("fe2s2-2_small") / "tensors_compressed" / "tensors_compressed_"
PHASE_BITS = 10 # number of qubits for phase gradient rotation

n_tensors = len(glob.glob(str(COMPRESSED_PATH) + "[0-9]*.npz"))
sparse_tensors = [load_npz(f"{COMPRESSED_PATH}{i}.npz") for i in range(n_tensors)]
dense_tensors = [t.toarray() for t in sparse_tensors]
mps_full = MPSWavefunction(dense_tensors)
print(f"MPSWavefunction: {mps_full.num_sites} sites, {mps_full.num_qubits} qubits")
print(f"Bond dims: {mps_full.bond_dims}")

MPSWavefunction: 20 sites, 40 qubits
Bond dims: [1, 4, 16, 64, 249, 742, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 871, 255, 64, 16, 4, 1]


## QDK-Chemistry MPS State Preparation

This branch implements 2 MPS state preparation algorithms: one regular and one optimized for sparsity of the matrices induced by symmetry.
We generate the circuit and run resource estimation.

In [ ]:
# Dense state preparation
algo_dense = MPSSequentialStatePreparation()
algo_dense.settings().set("rotation_bits", PHASE_BITS)
algo_dense.settings().set("fast_resource_estimation", True)

circuit_dense = algo_dense.run(mps_full)
result_dense = circuit_dense.estimate()
qdk_dense_counts = result_dense.logical_counts
print(f"\nQDK Dense logical counts: {qdk_dense_counts}")

# Sparse state preparation
algo_sparse = MPSSparseStatePreparation()
algo_sparse.settings().set("rotation_bits", PHASE_BITS)
algo_sparse.settings().set("fast_resource_estimation", True)

circuit_sparse = algo_sparse.run(mps_full)
result_sparse = circuit_sparse.estimate()
qdk_sparse_counts = result_sparse.logical_counts
print(f"QDK Sparse logical counts: {qdk_sparse_counts}")


QDK Dense logical counts: {'numQubits': 245, 'tCount': 54, 'rotationCount': 216, 'rotationDepth': 62, 'cczCount': 8068390, 'ccixCount': 0, 'measurementCount': 8457990}
QDK Sparse logical counts: {'numQubits': 296, 'tCount': 54, 'rotationCount': 216, 'rotationDepth': 62, 'cczCount': 345654, 'ccixCount': 0, 'measurementCount': 105092}


## Qualtran MPS State Preparation

In [4]:
base = Path("../MPSPrep/qualtran_impl")
for pkg in ["isometry/src", "unitary_synthesis/src", "state_preparation/src"]:
    p = str(base / pkg)
    if p not in sys.path:
        sys.path.insert(0, p)

from qualtran import QFxp
from qualtran.resource_counting import QECGatesCost, QubitCount, get_cost_value
from state_preparation import MPSPreparation, SparseThreeTensor

qt_tensors = tuple(SparseThreeTensor.from_dense(t) for t in dense_tensors)

# Dense state preparation
qt_dense = MPSPreparation(
    qt_tensors,
    QFxp(PHASE_BITS, PHASE_BITS),
    exploit_block_sparsity=False,
    is_complex=False,
    force_shaped=True,
)
qt_dense_qubits = int(get_cost_value(qt_dense, QubitCount()))
qt_dense_cost = get_cost_value(qt_dense, QECGatesCost())
qt_dense_toffoli = int(qt_dense_cost.and_bloq + qt_dense_cost.cswap)
print(f"\nQualtran Dense: {qt_dense_qubits} qubits, {qt_dense_toffoli} Toffoli")

# Sparse (block-symmetry-aware) state preparation
qt_sparse = MPSPreparation(
    qt_tensors,
    QFxp(PHASE_BITS, PHASE_BITS),
    exploit_block_sparsity=True,
    is_complex=False,
    force_shaped=True,
)
qt_sparse_qubits = int(get_cost_value(qt_sparse, QubitCount()))
qt_sparse_cost = get_cost_value(qt_sparse, QECGatesCost())
qt_sparse_toffoli = int(qt_sparse_cost.and_bloq + qt_sparse_cost.cswap)
print(f"Qualtran Sparse: {qt_sparse_qubits} qubits, {qt_sparse_toffoli} Toffoli")



Qualtran Dense: 218 qubits, 7165772 Toffoli
Qualtran Sparse: 251 qubits, 968296 Toffoli


## QREv3

In [ ]:
from qdk.qre import estimate
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

# Majorana-based architecture with an error rate of $10^{-5}$ and the `ThreeAux` 
# QEC code, along with round-based magic state factories
architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

# For QDK Chemistry circuit
app = circuit_dense.get_qre_application()
results = estimate(application=app, architecture, isa_query, max_error=0.01, name="MPS Dense")
display(results.as_frame())

,name,qubits,runtime,error,factories
0,MPS Dense,213896,0 days 00:26:40.550602,0.009118,10×T
1,MPS Dense,222264,0 days 00:21:46.571920,0.009100,12×T
2,MPS Dense,237400,0 days 00:17:25.257536,0.006469,16×T
3,MPS Dense,491696,0 days 00:17:25.255552,0.009815,21×T


In [ ]:
from qsharp.estimator import LogicalCounts
from qdk.qre.application import QSharpApplication

def get_bloq_logical_counts(bloq):
    num_qubits = bloq.signature.n_qubits()
    complexity = bloq.t_complexity()

    return LogicalCounts({
        "numQubits": num_qubits,
        "tCount": complexity.t,
        "rotationCount": complexity.rotations,
        "rotationDepth": complexity.rotations
    })

logical_counts = get_bloq_logical_counts(qt_dense)
results = estimate(QSharpApplication(logical_counts), architecture, isa_query, max_error=0.01, name="Qualtran Dense")
display(results.as_frame())

,name,qubits,runtime,error
0,Qualtran Dense,80681,0 days 00:32:19.634228,0.009453
1,Qualtran Dense,89049,0 days 00:26:23.374880,0.009448
2,Qualtran Dense,132189,0 days 00:25:43.790508,0.004110
3,Qualtran Dense,157305,0 days 00:21:06.699904,0.001832
